In [ ]:
pip install pennylane

In [ ]:
# ============================================================
# use these to suppres the warning that are generated during training--- this keep the train log clean and readable
# ============================================================
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

# TensorFlow internal warnings
import tensorflow as tf
tf.get_logger().setLevel("ERROR")

# PennyLane warnings
warnings.filterwarnings("ignore", module="pennylane")


In [ ]:
# ============================================================
# Imports
# ============================================================
import time
import pickle
import numpy as np
import pandas as pd
import pennylane as qml
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Embedding, Bidirectional, LSTM,
    Dense, Dropout, TimeDistributed,
    GlobalAveragePooling1D, Concatenate
)
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)

# ============================================================
# Reproducibility
# ============================================================
SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

# ============================================================
# Hyperparameters (MATCH CLASSICAL)
# ============================================================
MAX_SAMPLES = 500_000
VOCAB_SIZE  = 10_000
MAX_LEN     = 50
BATCH_SIZE  = 128
EPOCHS      = 20
LR          = 1e-4
N_QUBITS    = 5
N_RUNS      = 5

# ============================================================
# Load Tokenizer (FROM CLASSICAL PIPELINE)
# ============================================================
TOKENIZER_PATH = "/kaggle/input/classical-tokenizer/tokenizer.pkl"

with open(TOKENIZER_PATH, "rb") as f:
    tokenizer = pickle.load(f)

print("[INFO] Tokenizer loaded successfully")

def encode(texts):
    return pad_sequences(
        tokenizer.texts_to_sequences(texts),
        maxlen=MAX_LEN,
        padding="post",
        truncating="post"
    )

# ============================================================
# Dataset Loader
# ============================================================
def load_sentiment140(path):
    cols = ["target", "id", "date", "query", "user", "text"]
    df = pd.read_csv(path, encoding="latin-1", header=None, names=cols)
    df = df[["target", "text"]]
    df["target"] = df["target"].replace({4: 1})
    return df

FILE_PATH = "/kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv"
df = load_sentiment140(FILE_PATH)

df = (
    df.groupby("target", group_keys=False)
      .apply(lambda x: x.sample(MAX_SAMPLES // 2, random_state=SEED))
      .sample(frac=1.0, random_state=SEED)
      .reset_index(drop=True)
)

print(f"[INFO] Dataset size used: {len(df)}")

X = encode(df.text.values)
y = df.target.values

# ============================================================
# Train / Val / Test Split (60 / 20 / 20)
# ============================================================
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED
)

print(f"[INFO] Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# ============================================================
# Quantum CF2++ Layer (Pauli X/Y/Z)
# ============================================================
# ============================================================
# Quantum CF2++ Layer (Keras + TimeDistributed SAFE)
# ============================================================
class QuantumCF2Layer(tf.keras.layers.Layer):
    def __init__(self, n_qubits):
        super().__init__()
        self.n_qubits = n_qubits
        self.dev = qml.device("default.qubit", wires=n_qubits)

        @qml.qnode(self.dev, interface="tf")
        def circuit(x):
            # Angle embedding
            for i in range(n_qubits):
                qml.RY(x[i], wires=i)

            # Linear entanglement (CF2++)
            for i in range(n_qubits - 1):
                qml.CNOT(wires=[i, i + 1])

            # ONE observable per qubit (IMPORTANT)
            return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

        self.circuit = circuit

    def call(self, inputs):
        inputs = tf.math.l2_normalize(inputs, axis=-1)

        def quantum_eval(x):
            q_vals = self.circuit(x)          # complex128 (numerically)
            q_vals = tf.math.real(q_vals)     # ✅ physics-correct
            return tf.cast(q_vals, tf.float32)

        return tf.map_fn(
            quantum_eval,
            inputs,
            fn_output_signature=tf.TensorSpec(
                shape=(self.n_qubits,),
                dtype=tf.float32
            )
        )

    def compute_output_shape(self, input_shape):
        # Required for TimeDistributed
        return (*input_shape[:-1], self.n_qubits)



# ============================================================
# CF2++ Hybrid Quantum Model
# ============================================================
def build_quantum_cf2_model():
    inp = Input(shape=(MAX_LEN,))

    # --------------------------------------------------
    # Classical text encoder
    # --------------------------------------------------
    x = Embedding(VOCAB_SIZE, 128)(inp)
    x = Bidirectional(LSTM(128, return_sequences=True))(x)
    x = Dropout(0.5)(x)

    # Temporal compression (classical)
    x_pooled = GlobalAveragePooling1D()(x)   # (None, 256)

    # --------------------------------------------------
    # Quantum CF2++ projection (compressed)
    # --------------------------------------------------
    q = Dense(N_QUBITS, activation="tanh")(x_pooled)
    q = QuantumCF2Layer(N_QUBITS)(q)         # (None, N_QUBITS)

    # --------------------------------------------------
    # Fusion + classifier
    # --------------------------------------------------
    fused = Concatenate()([x_pooled, q])
    fused = Dense(64, activation="relu")(fused)
    out = Dense(2, activation="softmax")(fused)

    model = Model(inp, out)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(LR),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model


# ============================================================
# Training Runs
# ============================================================
all_metrics = []
histories = []
epoch_counts = []

for run in range(N_RUNS):
    print(f"\n==============================")
    print(f" QUANTUM CF2++ RUN {run+1}/{N_RUNS}")
    print("==============================")

    tf.keras.utils.set_random_seed(run)
    model = build_quantum_cf2_model()

    start = time.time()
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=[EarlyStopping(patience=3, restore_best_weights=True)],
        verbose=1
    )
    train_time = time.time() - start

    epochs_used = len(history.history["loss"])
    histories.append(history.history)
    epoch_counts.append(epochs_used)

    print(f"[INFO] Training stopped at epoch: {epochs_used}")

    y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)

    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)

    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")

    all_metrics.append({
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": prec,
        "recall": rec,
        "f1": f1_score(y_test, y_pred),
        "train_time_sec": train_time,
        "epochs_used": epochs_used
    })

# ============================================================
# Aggregate Metrics
# ============================================================
df_q_metrics = pd.DataFrame(all_metrics)

print("\n=== QUANTUM CF2++ TEST PERFORMANCE (MEAN ± STD) ===")
print(df_q_metrics.agg(["mean", "std"]))

# ============================================================
# Save Artifacts (MATCH CLASSICAL)
# ============================================================
df_q_metrics.to_csv("quantum_cf2_metrics.csv", index=False)
np.save("quantum_cf2_metrics.npy", df_q_metrics.to_dict())
np.save("quantum_cf2_histories.npy", histories)

model.save("quantum_cf2_model_final.keras")

# ============================================================
# Confusion Matrix (LAST RUN)
# ============================================================
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(5,4))
plt.imshow(cm, cmap="Blues")
plt.title("Hybrid Quantum CF2++ BiLSTM Confusion Matrix")
plt.colorbar()

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center", fontsize=12)

plt.xticks([0,1], ["Negative", "Positive"])
plt.yticks([0,1], ["Negative", "Positive"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, digits=4))

# ============================================================
# Inference Latency
# ============================================================
start = time.time()
_ = model.predict(X_test[:1000], verbose=0)
latency = (time.time() - start) / 1000
print(f"\nAvg inference latency per sample: {latency:.6f} seconds")
